In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain.chat_models import init_chat_model

In [2]:
@tool
def calculator(expression: str) -> str:
    """Perform basic calculations."""
    
    return str(eval(expression))

In [3]:
@tool
def delete_database(table_name: str) -> str:
    """Deletes a database table."""
    
    return f"Table '{table_name}' deleted."

In [4]:
class ToolGuardrailMiddleware(AgentMiddleware):
    def before_tool(self, tool, args, state, runtime):
        tool_name = tool.name
        print(f"\n[GUARDRAIL] Checking tool: {tool_name}")

        blocked_tools = [
            "delete_database"
        ]
        if tool_name in blocked_tools:
            raise Exception(
                f"Tool '{tool_name}' is blocked for safety reasons."
            )

        if tool_name == "calculator":
            expression = args.get("expression", "")
            dangerous_patterns = [
                "__import__",
                "os.system",
                "subprocess",
                "open("
            ]
            for pattern in dangerous_patterns:
                if pattern in expression:
                    raise Exception(
                        "Unsafe calculator expression detected."
                    )

        for key, value in args.items():
            if isinstance(value, str) and len(value) > 200:
                raise Exception(
                    "Tool input too long."
                )

        return None

In [5]:
    def after_tool(self, tool, result, state, runtime):
        print(f"[GUARDRAIL] Tool output: {result}")
        if "password" in str(result).lower():
            return "Sensitive information removed."
        return result

In [6]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [7]:
agent = create_agent(
    model=llm,
    tools=[
        calculator,
        delete_database
    ],
    middleware=[
        ToolGuardrailMiddleware()
    ]
)

In [8]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate 25 * 4"
        }
    ]
})

print(response["messages"][-1].content)

The answer is 100.


In [10]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Delete the employee database"
        }
    ]
})

print(response["messages"][-1].content)

I can delete the employee database, but please note that this action is irreversible. Would you like to proceed?
